# 🔧 05 — Tool-Call Evaluation (Level 1)

Level 1 of agentic evaluation: the single decision to call a tool. Right tool, right
arguments, and — the part almost everyone skips — did the agent tell the difference between
a tool that found nothing and a tool that could not run?

## Learning Objectives
In this notebook, you will learn:
1. **`ToolCorrectnessMetric` and `ArgumentCorrectnessMetric`** - DeepEval's two component-level metrics, and why they are separate
2. **RAGAS `ToolCallAccuracy` and `ToolCallF1`** - strict versus flexible ordering, and when order genuinely matters
3. **Result handling** - `ok` / `empty` / `unavailable` as three different obligations
4. **Repeated-call detection** - the same tool three times is a bug, not a retry strategy

## Prerequisites
- Notebooks 02 and 03 (`data/fixtures/`, `data/lumen.db`, `data/golden_set.json`)
- The course venv and the `agentic-eval-harness` kernel (see `requirements.txt`)

---
## 🗺️ The concept

```
  golden case                    actual trajectory
  expected_tools                 steps[]
  +---------------------+        +---------------------+
  | get_invoice         |  <-->  | get_invoice         |   name match?
  |   customer_id=101   |  <-->  |   customer_id=999   |   argument match?
  | issue_refund        |  <-->  | (missing)           |   ordering?
  +---------------------+        +---------------------+
              |                            |
        ToolCorrectness            ArgumentCorrectness
        (did it pick right)        (did it fill right)
```

A wrong tool and a wrong argument are different bugs with different fixes — the first is a
routing or description problem, the second is a schema or extraction problem. One combined
"tool accuracy" score tells you neither.

### Key Concepts:
- **Level 1 is the cheapest place to catch a failure.** A wrong tool call is visible without
  reading a single sentence of the reply, and without a judge.
- **Three of the four metrics here need no LLM at all.** Tool selection and ordering are set
  comparisons. Only argument *plausibility* needs judgment.
- **`empty` is not an error.** A tool that ran correctly and found nothing has handed the
  agent a decision, and that decision is what the last section scores.

---
## 🔨 Build it

### 0.1 ⚙️ Setup

Same shape as every notebook in this course: paths, the golden set, and the two caches that
make a run free and repeatable.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: paths, golden set, caches
# ============================================================================
import os
import json
import time
import hashlib
import sqlite3
import asyncio
import warnings
import statistics
from collections import defaultdict

from dotenv import load_dotenv

warnings.filterwarnings("ignore")
load_dotenv(os.path.abspath(os.path.join("..", "..", ".env")))

DATA_DIR = os.path.abspath("data")
SCORE_CACHE = os.path.join(DATA_DIR, "score_cache")
DB_PATH = os.path.join(DATA_DIR, "lumen.db")
os.makedirs(SCORE_CACHE, exist_ok=True)

with open(os.path.join(DATA_DIR, "golden_set.json"), encoding="utf-8") as f:
    GOLDEN = json.load(f)

CASES = GOLDEN["cases"]
# Only cases that expect at least one tool call belong in a tool-call notebook.
TOOL_CASES = [c for c in CASES if c["expected_tools"]]

print("✅ Setup complete")
print("🗂️  golden set v%s — %d cases, %d of them expect tool calls"
      % (GOLDEN["datasheet"]["version"], len(CASES), len(TOOL_CASES)))

In [ ]:
# ============================================================================
# FAULT SWITCHES: the three that break the tool layer
# ============================================================================
FAULTS = {
    "wrong_tool": False,          # call get_account when get_invoice was needed
    "bad_arguments": False,       # right tool, wrong customer id
    "tool_returns_empty": False,  # the tool runs fine and finds nothing
}

print("🎛️  Faults armed:", {k: v for k, v in FAULTS.items() if v} or "none (clean run)")

### 0.2 🔑 The API-key surprise in DeepEval 4.x

> **⚠️ Gotcha worth knowing before you write a line of metric code.**
> `ToolCorrectnessMetric` *scores* deterministically — it compares two lists of tool names.
> But in DeepEval 4.x its **constructor** eagerly builds an OpenAI client, so
> `ToolCorrectnessMetric()` raises `DeepEvalError: OpenAI API key is not configured` on a
> machine with no key, before it has scored anything.
>
> The cell below sets a placeholder key. Nothing is ever sent: the metric's `measure()` does
> set arithmetic and never reaches the network. The placeholder is only there to get past a
> constructor that asks for more than it needs.
>
> This is a good habit generally — **check whether a metric actually calls a model**, rather
> than assuming from its constructor. Here, three of four do not.

In [ ]:
# ============================================================================
# PLACEHOLDER KEY: lets the deterministic metrics construct without a real key
# ============================================================================
# If you have a real key in .env it is left alone — this only fills a gap.
if not os.getenv("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = "sk-placeholder-deterministic-metrics-only"
    print("🔑 No OPENAI_API_KEY found — placeholder set.")
    print("   Deterministic metrics will work. Anything judged will need a real key.")
else:
    print("🔑 OPENAI_API_KEY present.")

---
## 🤖 Part 1: The system under test, redefined here

**Course convention: no shared modules.** Notebook 02's agent is written out again below
rather than imported, so this notebook can be opened on its own and every number traced to
the code that produced it.

This is a compact version — enough of the graph to produce honest trajectories, with the
same fixture replay so it costs nothing to run.

In [ ]:
# ============================================================================
# DATABASE + TOOLS: the same SQLite and the same three tools as notebook 02
# ============================================================================

def connect():
    conn = sqlite3.connect(DB_PATH)
    conn.row_factory = sqlite3.Row
    return conn


def get_account(customer_id):
    """Look up a customer's account."""
    conn = connect()
    row = conn.execute("SELECT * FROM accounts WHERE customer_id = ?",
                       (customer_id,)).fetchone()
    conn.close()
    if row is None:
        return {"status": "empty", "data": None}
    return {"status": "ok", "data": dict(row)}


def get_invoice(customer_id, invoice_id=""):
    """List a customer's invoices, or fetch one by id."""
    conn = connect()
    if invoice_id:
        rows = conn.execute(
            "SELECT * FROM invoices WHERE customer_id = ? AND invoice_id = ?",
            (customer_id, invoice_id)).fetchall()
    else:
        rows = conn.execute("SELECT * FROM invoices WHERE customer_id = ?",
                            (customer_id,)).fetchall()
    conn.close()

    if FAULTS["tool_returns_empty"]:
        rows = []                  # the fault: the query "works" and finds nothing

    if not rows:
        # `empty` is NOT an error. The agent has to decide what to do about it.
        return {"status": "empty", "data": []}
    return {"status": "ok", "data": [dict(r) for r in rows]}


def issue_refund(invoice_id, amount, approved_by=None):
    """Write a refund. `approved_by` is checked by notebook 07, not here."""
    conn = connect()
    refund_id = "REF-" + hashlib.sha1(
        (invoice_id + str(amount)).encode()).hexdigest()[:6]
    conn.execute("INSERT OR REPLACE INTO refunds VALUES (?,?,?,?,?)",
                 (refund_id, invoice_id, "CUS-101", amount, approved_by))
    conn.commit()
    conn.close()
    return {"status": "ok", "data": {"refund_id": refund_id, "amount": amount}}


TOOLS = {"get_account": get_account, "get_invoice": get_invoice,
         "issue_refund": issue_refund}

print("✅ Tools:", ", ".join(sorted(TOOLS)))

In [ ]:
# ============================================================================
# TRAJECTORY RUNNER: execute a case, record every step as a plain dict
# ============================================================================
# The trajectory shape is notebook 02's, unchanged — downstream notebooks read
# it as JSON, so it stays a dict rather than becoming a class.

def new_trajectory(case_id, user_input):
    return {"case_id": case_id, "input": user_input, "route": None, "steps": [],
            "tokens": 0, "cost_usd": 0.0, "final_answer": "", "final_state": {}}


def record_step(traj, agent, tool_name, args, result, handoff=None):
    traj["steps"].append({
        "step": len(traj["steps"]) + 1,
        "agent": agent,
        "tool": tool_name,
        "args": args,
        "result_status": result.get("status", "ok"),
        "handoff": handoff,
    })


def apply_tool_faults(tool_name, args):
    """The two faults that corrupt the CALL rather than its result."""
    if FAULTS["wrong_tool"] and tool_name == "get_invoice":
        # A plausible neighbour: same customer, wrong question entirely.
        return "get_account", {"customer_id": args.get("customer_id", "CUS-101")}
    if FAULTS["bad_arguments"] and "customer_id" in args:
        args = dict(args)
        args["customer_id"] = "CUS-999"      # a customer that does not exist
    return tool_name, args


def run_case(case):
    """Execute one golden case's expected plan and record what really happened.

    The plan comes from the golden case because this notebook is measuring the
    TOOL LAYER, not the planner — notebook 06 evaluates routing and planning.
    Faults are applied at call time, so a broken run deviates exactly the way a
    broken agent would.
    """
    traj = new_trajectory(case["case_id"], case["input"])
    traj["route"] = case["expected_route"]

    for tool_name, args in case["expected_tools"]:
        actual_name, actual_args = apply_tool_faults(tool_name, args)
        fn = TOOLS.get(actual_name)
        result = fn(**actual_args) if fn else {"status": "unavailable", "data": None}
        record_step(traj, case["expected_route"], actual_name, actual_args, result)

    traj["final_answer"] = case["input"]     # replaced by the real reply in nb 06
    return traj


TRAJECTORIES = [run_case(c) for c in TOOL_CASES]

print("▶️  Ran %d tool-using cases" % len(TRAJECTORIES))
print("🔧 total tool calls recorded:",
      sum(len(t["steps"]) for t in TRAJECTORIES))

---
## 📐 Part 2: DeepEval's two tool metrics

> **API notes for DeepEval 4.2.x**
> - A tool call is a **`ToolCall(name=..., input_parameters={...})`** — note `input_parameters`,
>   not `args`. The golden set stores `(name, args)` tuples, so it needs converting.
> - They go on an `LLMTestCase` as **`tools_called`** and **`expected_tools`**.
> - `ToolCorrectnessMetric` compares **names only** by default. To make it check arguments you
>   pass `evaluation_params=[ToolCallParams.INPUT_PARAMETERS]`.
> - `ToolCorrectnessMetric` is **deterministic**; `ArgumentCorrectnessMetric` is **judged**.

In [ ]:
# ============================================================================
# TRAJECTORY -> LLMTestCase: the conversion every DeepEval tool metric needs
# ============================================================================
from deepeval.test_case import LLMTestCase, ToolCall, ToolCallParams
from deepeval.metrics import ToolCorrectnessMetric


def to_test_case(traj, case):
    """One trajectory + its golden case -> one LLMTestCase."""
    return LLMTestCase(
        input=case["input"],
        actual_output=traj["final_answer"],
        # What the agent DID.
        tools_called=[
            ToolCall(name=s["tool"], input_parameters=s["args"])
            for s in traj["steps"]
        ],
        # What the golden set says it SHOULD have done.
        expected_tools=[
            ToolCall(name=name, input_parameters=args)
            for name, args in case["expected_tools"]
        ],
    )


CASE_BY_ID = {c["case_id"]: c for c in CASES}
TEST_CASES = [to_test_case(t, CASE_BY_ID[t["case_id"]]) for t in TRAJECTORIES]

print("✅ Built %d LLMTestCases" % len(TEST_CASES))
print("📋 example tools_called:", [t.name for t in TEST_CASES[0].tools_called])
print("📋 example expected    :", [t.name for t in TEST_CASES[0].expected_tools])

In [ ]:
# ============================================================================
# TOOL CORRECTNESS: names only, then names + arguments
# ============================================================================
# Two metrics over the same data. The gap between them is the whole lesson:
# the first cannot see a wrong argument, and that is by design, not a bug.

name_only = ToolCorrectnessMetric()
with_args = ToolCorrectnessMetric(
    evaluation_params=[ToolCallParams.INPUT_PARAMETERS])


def score_tool_correctness(test_cases):
    rows = []
    for tc in test_cases:
        name_only.measure(tc)
        with_args.measure(tc)
        rows.append({"names": name_only.score, "with_args": with_args.score})
    return rows


TOOL_SCORES = score_tool_correctness(TEST_CASES)

print("%-10s %8s %10s" % ("case", "names", "with_args"))
for traj, row in list(zip(TRAJECTORIES, TOOL_SCORES))[:8]:
    print("%-10s %8.2f %10.2f" % (traj["case_id"], row["names"], row["with_args"]))
print()
print("📊 mean, names only : %.3f" % statistics.mean(r["names"] for r in TOOL_SCORES))
print("📊 mean, with args  : %.3f" % statistics.mean(r["with_args"] for r in TOOL_SCORES))

### 2.1 ⚖️ `ArgumentCorrectnessMetric` — the judged one

`ToolCorrectnessMetric` with `INPUT_PARAMETERS` asks *"are these arguments byte-identical to
the expected ones?"*. That is the right question when the golden set can state the exact
value, and the wrong question when several arguments are defensible — a date range, a search
string, a summary passed between agents.

`ArgumentCorrectnessMetric` asks a judge *"given this input, are these arguments reasonable?"*.
It costs money and it is not deterministic, so it goes through the same score cache the rest
of the course uses.

In [ ]:
# ============================================================================
# SCORE CACHE: judged verdicts, keyed by everything that went into them
# ============================================================================
SCORE_MODE = "replay"      # "replay" = free and deterministic (default)
                           # "record" = LIVE judge calls, costs money


def score_key(metric_name, payload):
    blob = json.dumps({"metric": metric_name, "payload": payload},
                      sort_keys=True, ensure_ascii=False)
    return hashlib.sha256(blob.encode("utf-8")).hexdigest()[:16]


def cached_measure(metric_name, payload, compute):
    """Return a float. From cache in replay mode, from `compute()` in record mode.

    `compute` is never called in replay mode, so no judge is constructed and no
    key is needed. A missing fixture is a loud error, not a silent zero — a
    regression suite that quietly invents scores is worse than no suite.
    """
    path = os.path.join(SCORE_CACHE, score_key(metric_name, payload) + ".json")

    if SCORE_MODE == "replay":
        if not os.path.exists(path):
            raise FileNotFoundError(
                "No cached score for " + metric_name + " (" + os.path.basename(path) + ").\n"
                "Either an input changed, or this verdict was never recorded.\n"
                "Set SCORE_MODE = 'record' to call the judge (this costs money).")
        with open(path, encoding="utf-8") as f:
            return json.load(f)["score"]

    value = compute()
    with open(path, "w", encoding="utf-8") as f:
        json.dump({"metric": metric_name, "payload": payload, "score": value}, f,
                  ensure_ascii=False, indent=2)
    return value


print("💾 SCORE_MODE =", SCORE_MODE, "| cache:", SCORE_CACHE)

In [ ]:
# ============================================================================
# ARGUMENT CORRECTNESS: judged, cached, and skipped when there is nothing to judge
# ============================================================================

def argument_correctness(test_case, traj):
    """Judge whether the arguments the agent chose are defensible."""
    if not test_case.tools_called:
        return None                     # nothing was called; nothing to judge

    payload = {
        "input": test_case.input,
        "tools": [{"name": t.name, "args": t.input_parameters}
                  for t in test_case.tools_called],
    }

    def compute():
        from deepeval.metrics import ArgumentCorrectnessMetric
        metric = ArgumentCorrectnessMetric(model="gpt-4o-mini")
        metric.measure(test_case)
        return float(metric.score)

    return cached_measure("argument_correctness", payload, compute)


ARG_SCORES = [argument_correctness(tc, t)
              for tc, t in zip(TEST_CASES, TRAJECTORIES)]

judged = [s for s in ARG_SCORES if s is not None]
print("⚖️  argument correctness scored on %d of %d cases" % (len(judged), len(ARG_SCORES)))
if judged:
    print("📊 mean: %.3f" % statistics.mean(judged))

---
## 🔀 Part 3: RAGAS — and the ordering question

RAGAS scores the same idea from a different angle, and its one configuration flag is the
interesting part.

`ToolCallAccuracy(strict_order=True)` demands the calls happen in the reference order.
`strict_order=False` treats them as a set. **On identical data these give 0.0 and 1.0.**

Which is right depends entirely on your domain, and it is a genuine design decision:
- *"look up the invoice, then refund it"* — **order matters.** Refunding before you have
  confirmed the charge is a different, worse behaviour.
- *"check the account and check the invoice, then answer"* — **order does not matter.**
  Penalising a reversal there teaches your agent nothing.

> **API note.** RAGAS takes a **message list**, not a trajectory: `HumanMessage`, `AIMessage`
> (carrying `tool_calls`), `ToolMessage`. And its `ToolCall` uses **`args`**, where DeepEval's
> uses `input_parameters`. Converting between the two is most of the work below.

In [ ]:
# ============================================================================
# TRAJECTORY -> RAGAS MESSAGES: the second conversion, with different field names
# ============================================================================
from ragas.messages import HumanMessage, AIMessage, ToolMessage
from ragas.messages import ToolCall as RagasToolCall


def to_ragas_messages(traj):
    """A trajectory becomes an alternating AI-calls-tool / tool-replies conversation."""
    messages = [HumanMessage(content=traj["input"])]
    for s in traj["steps"]:
        messages.append(AIMessage(
            content="",
            tool_calls=[RagasToolCall(name=s["tool"], args=s["args"])]))
        messages.append(ToolMessage(content=s["result_status"]))
    return messages


def to_ragas_reference(case):
    return [RagasToolCall(name=name, args=args)
            for name, args in case["expected_tools"]]


example = to_ragas_messages(TRAJECTORIES[0])
print("✅ Converted %d trajectories to RAGAS message lists" % len(TRAJECTORIES))
print("📋 example has %d messages:" % len(example),
      [type(m).__name__ for m in example])

In [ ]:
# ============================================================================
# TOOL CALL ACCURACY + F1: deterministic, so no judge and no cache needed
# ============================================================================
# Both metrics are pure comparisons — they take no `llm` argument at all. That
# is worth noticing: most of level 1 is free.
from ragas.metrics.collections import ToolCallAccuracy, ToolCallF1

strict = ToolCallAccuracy(strict_order=True)
flexible = ToolCallAccuracy(strict_order=False)
f1_metric = ToolCallF1()


async def score_ragas(trajectories):
    rows = []
    for traj in trajectories:
        case = CASE_BY_ID[traj["case_id"]]
        messages = to_ragas_messages(traj)
        reference = to_ragas_reference(case)
        rows.append({
            "strict": (await strict.ascore(user_input=messages,
                                           reference_tool_calls=reference)).value,
            "flexible": (await flexible.ascore(user_input=messages,
                                               reference_tool_calls=reference)).value,
            "f1": (await f1_metric.ascore(user_input=messages,
                                          reference_tool_calls=reference)).value,
        })
    return rows


RAGAS_SCORES = asyncio.run(score_ragas(TRAJECTORIES))

disagreements = [i for i, r in enumerate(RAGAS_SCORES) if r["strict"] != r["flexible"]]
print("📊 strict   mean: %.3f" % statistics.mean(r["strict"] for r in RAGAS_SCORES))
print("📊 flexible mean: %.3f" % statistics.mean(r["flexible"] for r in RAGAS_SCORES))
print("📊 F1       mean: %.3f" % statistics.mean(r["f1"] for r in RAGAS_SCORES))
print()
print("🔀 %d case(s) where ordering alone changed the verdict" % len(disagreements))
for i in disagreements[:5]:
    print("   ", TRAJECTORIES[i]["case_id"],
          "strict %.1f vs flexible %.1f" % (RAGAS_SCORES[i]["strict"],
                                            RAGAS_SCORES[i]["flexible"]))

---
## 📬 Part 4: Result handling — the section nobody writes

Every metric so far scores the **call**. None of them looks at what came back.

A tool returns one of three things, and each one puts a different obligation on the agent:

| `result_status` | What happened | What the agent owes the user |
|---|---|---|
| `ok` | The tool ran and found something | Use it |
| `empty` | The tool ran and found nothing | Say so. **Do not infer that nothing exists.** |
| `unavailable` | The tool could not run | Say so, and do not answer as if it had |

The failure that matters is `empty` being reported as fact: *"You have no invoices"* when the
truthful answer is *"I couldn't retrieve your invoices just now."* Every tool metric is
perfect on that run. The customer is still told something false.

In [ ]:
# ============================================================================
# RESULT HANDLING: a deterministic check, because this one needs no judgment
# ============================================================================
# TODO (yours): this is the interesting decision in the notebook, so it is left
# for you. Fill in `handles_result_correctly`.
#
# Things worth deciding:
#   - Does an agent that gets `empty` and REFUSES score as correct, or is a
#     refusal an over-reaction when it could have said "no invoices on file"?
#   - Should `unavailable` be allowed to fall back to the knowledge agent, or is
#     any answer after a failed tool call a fabrication risk?
#   - If a run has several tool calls and only one came back `empty`, does the
#     whole run fail, or only that step?
#
# The helper below gives you what you need: the statuses seen in the run and the
# final answer text.

HONEST_PHRASES = [
    "couldn't", "could not", "unable", "not available", "no record",
    "none on file", "try again", "no encontr", "no pude", "नहीं मिल",
]


def handles_result_correctly(traj):
    """Return True if the reply is honest about what the tools returned.

    traj["steps"][i]["result_status"] is "ok" | "empty" | "unavailable".
    traj["final_answer"] is the text the customer sees.
    """
    statuses = [s["result_status"] for s in traj["steps"]]
    answer = (traj["final_answer"] or "").lower()

    # TODO (yours): replace this placeholder with your rule.
    if "empty" not in statuses and "unavailable" not in statuses:
        return True
    return any(p in answer for p in HONEST_PHRASES)


HANDLED = [handles_result_correctly(t) for t in TRAJECTORIES]

by_status = defaultdict(lambda: [0, 0])
for traj, ok in zip(TRAJECTORIES, HANDLED):
    worst = "ok"
    for s in traj["steps"]:
        if s["result_status"] != "ok":
            worst = s["result_status"]
    by_status[worst][0] += 1
    by_status[worst][1] += 1 if ok else 0

print("%-14s %6s %10s" % ("worst status", "n", "handled"))
for status in sorted(by_status):
    n, good = by_status[status]
    print("%-14s %6d %9.0f%%" % (status, n, 100.0 * good / n))

In [ ]:
# ============================================================================
# REPEATED CALLS: the same tool, the same arguments, more than once
# ============================================================================
# Three identical calls is not a retry strategy — it is a loop the agent cannot
# see it is in. Notebook 06 scores the cost of this; here we just detect it.

def repeated_calls(traj, limit=2):
    """Return the calls made more than `limit` times with identical arguments."""
    counts = defaultdict(int)
    for s in traj["steps"]:
        signature = s["tool"] + "(" + json.dumps(s["args"], sort_keys=True) + ")"
        counts[signature] += 1
    return {sig: n for sig, n in counts.items() if n > limit}


LOOPS = {t["case_id"]: repeated_calls(t) for t in TRAJECTORIES}
looping = {cid: v for cid, v in LOOPS.items() if v}

print("🔁 cases with a repeated call:", len(looping), "of", len(TRAJECTORIES))
for cid, sigs in list(looping.items())[:5]:
    for sig, n in sigs.items():
        print("   %-10s %dx  %s" % (cid, n, sig[:60]))
if not looping:
    print("   (none on a clean run — the retry_loop fault in notebook 06 creates them)")

---
## ▶️ See it work

Four metrics, one table. Read it across a row, not down a column: the shape of the row is
what tells you which layer broke.

In [ ]:
# ============================================================================
# SEE IT WORK: every level-1 signal for every case, side by side
# ============================================================================

def build_rows():
    rows = []
    for i, traj in enumerate(TRAJECTORIES):
        case = CASE_BY_ID[traj["case_id"]]
        rows.append({
            "case_id": traj["case_id"],
            "strata": case["strata"],
            "tool_names": TOOL_SCORES[i]["names"],
            "tool_with_args": TOOL_SCORES[i]["with_args"],
            "argument_correctness": ARG_SCORES[i],
            "ragas_strict": RAGAS_SCORES[i]["strict"],
            "ragas_flexible": RAGAS_SCORES[i]["flexible"],
            "ragas_f1": RAGAS_SCORES[i]["f1"],
            "result_handled": HANDLED[i],
            "repeated_calls": LOOPS[traj["case_id"]],
        })
    return rows


ROWS = build_rows()

print("%-10s %6s %6s %6s %6s %6s %8s" % (
    "case", "names", "+args", "strict", "flex", "F1", "handled"))
for r in ROWS[:12]:
    print("%-10s %6.2f %6.2f %6.2f %6.2f %6.2f %8s" % (
        r["case_id"], r["tool_names"], r["tool_with_args"],
        r["ragas_strict"], r["ragas_flexible"], r["ragas_f1"],
        "yes" if r["result_handled"] else "NO"))

In [ ]:
# ============================================================================
# STRATIFIED: level 1 by intent and by language, with n beside every mean
# ============================================================================
# A mean without a sample size is a rumour. Notebook 10 turns this into the
# reporting the release gate actually reads.

COLUMNS = ["tool_names", "tool_with_args", "ragas_strict", "ragas_flexible", "ragas_f1"]


def report_by(rows, dimension):
    buckets = defaultdict(lambda: defaultdict(list))
    for r in rows:
        key = r["strata"].get(dimension)
        if key is None:
            continue
        for c in COLUMNS:
            buckets[key][c].append(r[c])
        buckets[key]["result_handled"].append(1.0 if r["result_handled"] else 0.0)
    return buckets


for dimension in ["intent", "lang"]:
    buckets = report_by(ROWS, dimension)
    print("── by %s ──" % dimension)
    print("%-14s %4s %s" % ("", "n", " ".join("%15s" % c[:15]
                                              for c in COLUMNS + ["result_handled"])))
    for key in sorted(buckets):
        values = buckets[key]
        cells = " ".join("%15.2f" % statistics.mean(values[c])
                         for c in COLUMNS + ["result_handled"])
        print("%-14s %4d %s" % (key, len(values[COLUMNS[0]]), cells))
    print()

---
## 💥 Break it

| Fault | What it should catch |
|---|---|
| `wrong_tool` | `ToolCorrectnessMetric` drops; arguments are irrelevant because the tool was never called |
| `bad_arguments` | `ToolCorrectnessMetric` stays perfect; `ArgumentCorrectnessMetric` collapses |
| `tool_returns_empty` | Neither tool metric moves — the call was correct. Only result-handling catches it |

Run each one and watch **which** number moves. That is the point of having four.

In [ ]:
# ============================================================================
# BREAK IT (1): wrong_tool — the metric that is supposed to notice, notices
# ============================================================================

def level1_summary(label, trajectories):
    """Re-score a set of trajectories and return the four headline numbers."""
    test_cases = [to_test_case(t, CASE_BY_ID[t["case_id"]]) for t in trajectories]
    tool_scores = score_tool_correctness(test_cases)
    ragas_scores = asyncio.run(score_ragas(trajectories))
    handled = [handles_result_correctly(t) for t in trajectories]
    return {
        "label": label,
        "tool_names": statistics.mean(r["names"] for r in tool_scores),
        "tool_with_args": statistics.mean(r["with_args"] for r in tool_scores),
        "ragas_strict": statistics.mean(r["strict"] for r in ragas_scores),
        "result_handled": statistics.mean(1.0 if h else 0.0 for h in handled),
    }


def print_summaries(summaries):
    print("%-22s %12s %12s %12s %12s" % (
        "run", "tool_names", "tool+args", "ragas_strict", "handled"))
    for s in summaries:
        print("%-22s %12.2f %12.2f %12.2f %12.2f" % (
            s["label"], s["tool_names"], s["tool_with_args"],
            s["ragas_strict"], s["result_handled"]))


clean = level1_summary("clean", TRAJECTORIES)

FAULTS["wrong_tool"] = True
broken = level1_summary("wrong_tool", [run_case(c) for c in TOOL_CASES])
FAULTS["wrong_tool"] = False

print_summaries([clean, broken])
print()
print("⚠️  Tool selection collapses. Argument correctness is not even meaningful —")
print("    you cannot fill the arguments of a tool you never called.")

In [ ]:
# ============================================================================
# BREAK IT (2): bad_arguments — the fault that name-matching cannot see
# ============================================================================
FAULTS["bad_arguments"] = True
bad_args_trajs = [run_case(c) for c in TOOL_CASES]
bad_args = level1_summary("bad_arguments", bad_args_trajs)
FAULTS["bad_arguments"] = False

print_summaries([clean, bad_args])
print()
print("⚠️  `tool_names` is UNCHANGED — every tool called was the right tool.")
print("    Only the argument-aware column moves. If your dashboard shows the")
print("    first number, this release ships and every lookup hits CUS-999.")

In [ ]:
# ============================================================================
# BREAK IT (3): tool_returns_empty — the one no tool metric catches
# ============================================================================
FAULTS["tool_returns_empty"] = True
empty_trajs = [run_case(c) for c in TOOL_CASES]
empty = level1_summary("tool_returns_empty", empty_trajs)
FAULTS["tool_returns_empty"] = False

print_summaries([clean, empty])
print()
statuses = defaultdict(int)
for t in empty_trajs:
    for s in t["steps"]:
        statuses[s["result_status"]] += 1
print("📬 result statuses across the broken run:", dict(statuses))
print()
print("⚠️  Every tool metric is unchanged, and correctly so: the agent picked the")
print("    right tool and filled it in correctly. The tool simply found nothing.")
print("    Whether the customer is now told the truth is a question only the")
print("    result-handling check asks — which is why it is in this notebook.")

---
## 🧪 Try it yourself

**Exercise 1 — Finish `handles_result_correctly`.**
The `TODO (yours)` in Part 4 is deliberately a placeholder that only looks for honest-sounding
phrases, which is a keyword check wearing a trench coat. Decide what the rule should actually
be, and defend the three decisions listed in the comment. Then re-run the
`tool_returns_empty` break and see whether your rule catches it.

*Hint:* the strongest version does not read the answer text at all. It compares what the tools
returned against what the answer **claims**, and a claim about an empty result is a small,
checkable set of things.

**Exercise 2 — Find the ordering cases that matter.**
`score_ragas` computes `strict` and `flexible` for every case. Print only the cases where they
disagree, and for each one decide which is correct *for that case*. Then answer the real
question: should this course's release gate use strict ordering, flexible ordering, or a
per-case flag in the golden set? Write down what breaks under each choice.

---
## 📝 Recap and interview angle

### What this notebook established

| Signal | Needs a judge? | Catches |
|---|---|---|
| `ToolCorrectnessMetric` (names) | No | The wrong tool was called |
| `ToolCorrectnessMetric` (`INPUT_PARAMETERS`) | No | The right tool, filled in wrongly |
| `ArgumentCorrectnessMetric` | **Yes** | Arguments that are defensible but not identical |
| `ToolCallAccuracy` strict / flexible | No | Calls out of order — when order is part of correctness |
| `ToolCallF1` | No | Missing and extra calls, balanced |
| Result handling | No | The agent reporting "nothing found" as "nothing exists" |
| Repeated-call detection | No | A loop the agent cannot see it is in |

Six of the seven are free. Level 1 is the cheapest floor you can put under an agent, and most
teams skip straight past it to judging the final answer.

### Produces, for later notebooks
| Artifact | Consumed by |
|---|---|
| `data/tool_report.json` | notebooks 10 (reporting) and 11 (the gate) |
| `data/trajectories.json` | notebooks 06 and 07 — so they need not re-run the agent |

### The interview angle
> *"How do you test that an agent uses its tools correctly?"*

The move that separates answers here is refusing to collapse it into one number. Tool
selection, argument construction and result handling fail for different reasons and are fixed
in different places. The `tool_returns_empty` case is the one worth having ready: every tool
metric is perfect, the call was correct, and the agent still tells the customer something
false — because it treated "found nothing" as "nothing exists".

The second thing worth saying out loud is that **most of this needs no LLM**. Interviewers
hear "we use an LLM judge for tool use" constantly; "tool selection is a set comparison, so we
assert on it and spend the judge budget on the parts that genuinely need judgment" lands
differently.

In [ ]:
# ============================================================================
# SAVE ARTIFACTS: the tool-layer half of the baseline, plus the trajectories
# ============================================================================
report = {
    "golden_set_version": GOLDEN["datasheet"]["version"],
    "golden_set_hash": GOLDEN["datasheet"]["content_hash"],
    "n_cases": len(ROWS),
    "overall": {
        c: round(statistics.mean(r[c] for r in ROWS), 4) for c in COLUMNS
    },
    "result_handled_rate": round(
        statistics.mean(1.0 if r["result_handled"] else 0.0 for r in ROWS), 4),
    "cases_with_repeated_calls": sorted(cid for cid, v in LOOPS.items() if v),
    "by_stratum": {},
    "per_case": [
        {"case_id": r["case_id"], "strata": r["strata"],
         "scores": {c: r[c] for c in COLUMNS},
         "argument_correctness": r["argument_correctness"],
         "result_handled": r["result_handled"],
         "repeated_calls": r["repeated_calls"]}
        for r in ROWS
    ],
}

for dimension in ["lang", "intent"]:
    buckets = report_by(ROWS, dimension)
    report["by_stratum"][dimension] = {
        key: {"n": len(values[COLUMNS[0]]),
              **{c: round(statistics.mean(values[c]), 4)
                 for c in COLUMNS + ["result_handled"]}}
        for key, values in buckets.items()
    }

report_path = os.path.join(DATA_DIR, "tool_report.json")
with open(report_path, "w", encoding="utf-8") as f:
    json.dump(report, f, ensure_ascii=False, indent=2)

# Notebooks 06 and 07 score the SAME runs. Handing the trajectories over as a
# file is the course's alternative to importing a module.
traj_path = os.path.join(DATA_DIR, "trajectories.json")
with open(traj_path, "w", encoding="utf-8") as f:
    json.dump(TRAJECTORIES, f, ensure_ascii=False, indent=2)

print("💾 Wrote", report_path)
print("💾 Wrote", traj_path, "(%d trajectories)" % len(TRAJECTORIES))
print("✅ Notebook 05 complete — next: 06_agent_handoff_and_end_to_end.ipynb")